**DL_PA2   Roll_Number 25280103:     In this we have to check the model accurac by using the different optimization and genalization techniques: And try to increase the model accuracy by using different methods and techniques**

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [4]:
##Loading the data
train_data = np.load("processed_data/quickdraw_train.npz")
test_data = np.load("processed_data/quickdraw_test.npz")

print("Train keys:", train_data.files)
print("Test keys:", test_data.files)

Train keys: ['x_train', 'y_train', 'class_names']
Test keys: ['test_images']


In [5]:
#extract Arrays
# Extract arrays correctly

X_train = train_data['x_train']
y_train = train_data['y_train']
class_names = train_data['class_names']

X_test = test_data['test_images']

print("Train shape:", X_train.shape)
print("Labels shape:", y_train.shape)
print("Test shape:", X_test.shape)
print("Number of classes:", len(class_names))

Train shape: (60000, 784)
Labels shape: (60000,)
Test shape: (15000, 784)
Number of classes: 15


In [6]:
#preprocessing the data for training and testing
# Normalize pixel values to [0,1]

X_train = X_train / 255.0
X_test = X_test / 255.0

print("Min value:", X_train.min())
print("Max value:", X_train.max())

Min value: 0.0
Max value: 1.0


In [7]:
# Convert to PyTorch tensors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)

print("Tensor shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)

Using device: cpu
Tensor shapes:
X_train: torch.Size([60000, 784])
y_train: torch.Size([60000])
X_test: torch.Size([15000, 784])


In [8]:
# Create dataset
dataset = TensorDataset(X_train, y_train)

# 80% train, 20% validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)

print("Train size:", train_size)
print("Validation size:", val_size)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train size: 48000
Validation size: 12000
Train batches: 375
Validation batches: 94


""Part 1 Pencake Wise Model where we used the multiple neurons but less number of hidden layers""

In [9]:
# PART A — Pancake Model (Wide, Shallow)

class PancakeMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 15)
        )

    def forward(self, x):
        return self.model(x)

pancake = PancakeMLP().to(device)

# Count parameters
total_params = sum(p.numel() for p in pancake.parameters())
print("Pancake parameter count:", total_params)

Pancake parameter count: 1868815


In [10]:
def train_model(model, epochs=35):
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    train_acc_list = []
    val_acc_list = []

    for epoch in range(epochs):
        model.train()
        correct = 0
        total = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

        train_acc = correct / total

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                outputs = model(xb)
                preds = torch.argmax(outputs, dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)

        val_acc = correct / total

        train_acc_list.append(train_acc)
        val_acc_list.append(val_acc)

        print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}")

    return train_acc_list, val_acc_list

In [11]:
print("Training Pancake Model...")

p_train, p_val = train_model(pancake, epochs=35)                                                                                                                

Training Pancake Model...
Epoch 1: Train=0.6316, Val=0.7044
Epoch 2: Train=0.7357, Val=0.7466
Epoch 3: Train=0.7859, Val=0.7590
Epoch 4: Train=0.8162, Val=0.7718
Epoch 5: Train=0.8483, Val=0.7676
Epoch 6: Train=0.8751, Val=0.7594
Epoch 7: Train=0.9004, Val=0.7595
Epoch 8: Train=0.9208, Val=0.7585
Epoch 9: Train=0.9385, Val=0.7566
Epoch 10: Train=0.9530, Val=0.7590
Epoch 11: Train=0.9618, Val=0.7505
Epoch 12: Train=0.9663, Val=0.7582
Epoch 13: Train=0.9732, Val=0.7613
Epoch 14: Train=0.9757, Val=0.7599
Epoch 15: Train=0.9757, Val=0.7517
Epoch 16: Train=0.9762, Val=0.7578
Epoch 17: Train=0.9836, Val=0.7518
Epoch 18: Train=0.9818, Val=0.7544
Epoch 19: Train=0.9817, Val=0.7521
Epoch 20: Train=0.9859, Val=0.7516
Epoch 21: Train=0.9828, Val=0.7553
Epoch 22: Train=0.9839, Val=0.7525
Epoch 23: Train=0.9857, Val=0.7587
Epoch 24: Train=0.9876, Val=0.7539
Epoch 25: Train=0.9861, Val=0.7562
Epoch 26: Train=0.9871, Val=0.7509
Epoch 27: Train=0.9840, Val=0.7606
Epoch 28: Train=0.9894, Val=0.7602
Epo

**As we used the higher number of so features are memorised by the neurons and in this pancake as multiple nodes used so its giving the less accuracy than the training model sever overfitting....**

In [ ]:
#tower model
class TowerMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 15)
        )

    def forward(self, x):
        return self.model(x)

tower = TowerMLP()
print("Tower parameters:", sum(p.numel() for p in tower.parameters()))

In [ ]:
#train
t_train, t_val = train_model(tower, epochs=35)

In [ ]:
#Champion model
class ChampionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 768),
            nn.BatchNorm1d(768),
            nn.GELU(),
            nn.Dropout(0.35),

            nn.Linear(768, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Linear(512, 15)
        )

    def forward(self, x):
        return self.model(x)

champion = ChampionMLP()
print("Champion parameters:", sum(p.numel() for p in champion.parameters()))

In [ ]:
#train
c_train, c_val = train_model(champion, epochs=35)

In [ ]:
#Plot Curves
plt.plot(c_train, label="Train")
plt.plot(c_val, label="Val")
plt.legend()
plt.show()

In [ ]:
#Confusion Matrix
champion.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = champion(xb)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.show()

In [ ]:
#save the bestModel
torch.save(champion.state_dict(), "best_model.pth")

In [ ]:
#Generate test prediction
champion.eval()
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

with torch.no_grad():
    outputs = champion(X_test_tensor)
    preds = torch.argmax(outputs, dim=1)

np.savetxt("submission.csv", preds.numpy(), fmt="%d", delimiter=",")